In [1]:
# import the libraries
import pandas as pd
import sqlite3

In [2]:
# load the Excel file
df = pd.read_excel("Financial Sample.xlsx")

In [3]:
# number of rows and columns
print(df.shape)

(700, 16)


In [4]:
# list of column names
print(df.columns.tolist())

['Segment', 'Country', 'Product', 'Discount Band', 'Units Sold', 'Manufacturing Price', 'Sale Price', 'Gross Sales', 'Discounts', ' Sales', 'COGS', 'Profit', 'Date', 'Month Number', 'Month Name', 'Year']


In [5]:
# data type of each column (money columns should be numeric)
df.dtypes

Segment                        object
Country                        object
Product                        object
Discount Band                  object
Units Sold                    float64
Manufacturing Price             int64
Sale Price                      int64
Gross Sales                   float64
Discounts                     float64
 Sales                        float64
COGS                          float64
Profit                        float64
Date                   datetime64[ns]
Month Number                    int64
Month Name                     object
Year                            int64
dtype: object

In [6]:
# count of missing values per column
df.isna().sum()

Segment                 0
Country                 0
Product                 0
Discount Band          53
Units Sold              0
Manufacturing Price     0
Sale Price              0
Gross Sales             0
Discounts               0
 Sales                  0
COGS                    0
Profit                  0
Date                    0
Month Number            0
Month Name              0
Year                    0
dtype: int64

In [7]:
# preview the first 5 rows
df.head()

,Segment,Country,Product,Discount Band,Units Sold,Manufacturing Price,Sale Price,Gross Sales,Discounts,Sales,COGS,Profit,Date,Month Number,Month Name,Year
0,Government,Canada,Carretera,NaN,1618.5,3,20,32370.0,0.0,32370.0,16185.0,16185.0,2014-01-01,1,January,2014
1,Government,Germany,Carretera,NaN,1321.0,3,20,26420.0,0.0,26420.0,13210.0,13210.0,2014-01-01,1,January,2014
2,Midmarket,France,Carretera,NaN,2178.0,3,15,32670.0,0.0,32670.0,21780.0,10890.0,2014-06-01,6,June,2014
3,Midmarket,Germany,Carretera,NaN,888.0,3,15,13320.0,0.0,13320.0,8880.0,4440.0,2014-06-01,6,June,2014
4,Midmarket,Mexico,Carretera,NaN,2470.0,3,15,37050.0,0.0,37050.0,24700.0,12350.0,2014-06-01,6,June,2014


In [8]:
# clean column names: lowercase with underscores
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

In [9]:
# convert the date column to a real datetime
df["date"] = pd.to_datetime(df["date"])

In [10]:
# confirm the new column names and types
df.dtypes

segment                        object
country                        object
product                        object
discount_band                  object
units_sold                    float64
manufacturing_price             int64
sale_price                      int64
gross_sales                   float64
discounts                     float64
sales                         float64
cogs                          float64
profit                        float64
date                   datetime64[ns]
month_number                    int64
month_name                     object
year                            int64
dtype: object

In [11]:
# connect to (or create) the SQLite database file
conn = sqlite3.connect("financials.db")

In [12]:
# save the dataframe as a table called financials
df.to_sql("financials", conn, if_exists="replace", index=False)

700

In [13]:
# check the distinct dates (need clean monthly dates for Day 2)
pd.read_sql("SELECT DISTINCT date FROM financials ORDER BY date", conn)

,date
0,2013-09-01 00:00:00
1,2013-10-01 00:00:00
2,2013-11-01 00:00:00
3,2013-12-01 00:00:00
4,2014-01-01 00:00:00
5,2014-02-01 00:00:00
6,2014-03-01 00:00:00
7,2014-04-01 00:00:00
8,2014-05-01 00:00:00
9,2014-06-01 00:00:00


In [14]:
# total sales and profit by country
pd.read_sql("""
SELECT country, SUM(sales) AS total_sales, SUM(profit) AS total_profit
FROM financials
GROUP BY country
ORDER BY total_sales DESC
""", conn)

,country,total_sales,total_profit
0,United States of America,2.502983e+07,2995540.665
1,Canada,2.488765e+07,3529228.885
2,France,2.435417e+07,3781020.780
3,Germany,2.350534e+07,3680388.820
4,Mexico,2.094935e+07,2907523.110


In [15]:
# total profit by product
pd.read_sql("""
SELECT product, SUM(profit) AS total_profit
FROM financials
GROUP BY product
ORDER BY total_profit DESC
""", conn)

,product,total_profit
0,Paseo,4797437.950
1,VTT,3034608.020
2,Amarilla,2814104.060
3,Velo,2305992.465
4,Montana,2114754.880
5,Carretera,1826804.885


In [16]:
# total sales by segment
pd.read_sql("""
SELECT segment, SUM(sales) AS total_sales
FROM financials
GROUP BY segment
""", conn)

,segment,total_sales
0,Channel Partners,1.800594e+06
1,Enterprise,1.961169e+07
2,Government,5.250426e+07
3,Midmarket,2.381883e+06
4,Small Business,4.242792e+07
